### Automated data-quality checks.

In [0]:
spark.sql("create schema if not exists ecommerce_project.audit")

In [0]:
from pyspark.sql import functions as F

orders = spark.table("ecommerce_project.silver.orders")
order_items = spark.table("ecommerce_project.silver.order_items")
fact_sales = spark.table("ecommerce_project.gold.fact_sales")
session_funnel = spark.table("ecommerce_project.gold.session_funnel")
sessions = spark.table("ecommerce_project.silver.website_sessions")

checks = []


# Check if any Null order IDs
null_order_ids = orders.filter(
    F.col("order_id").isNull()
).count()

checks.append((
    "Orders: null primary keys",
    null_order_ids,
    0,
    "PASS" if null_order_ids == 0 else "FAIL"
))


# Duplicate order IDs
duplicate_order_ids = (
    orders.groupBy("order_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

checks.append((
    "Orders: duplicate primary keys",
    duplicate_order_ids,
    0,
    "PASS" if duplicate_order_ids == 0 else "FAIL"
))


# Check is any product price is zero or negative
invalid_prices = order_items.filter(
    F.col("price_usd") <= 0
).count()

checks.append((
    "Order items: invalid prices",
    invalid_prices,
    0,
    "PASS" if invalid_prices == 0 else "FAIL"
))


# Silver-order_items and Gold fact_sales: row-count validation
silver_item_count = order_items.count()
gold_sales_count = fact_sales.count()

checks.append((
    "Silver items vs Gold sales",
    gold_sales_count,
    silver_item_count,
    "PASS" if gold_sales_count == silver_item_count else "FAIL"
))


# Session row-count validation
silver_session_count = sessions.count()
gold_session_count = session_funnel.count()

checks.append((
    "Silver sessions vs Gold funnel",
    gold_session_count,
    silver_session_count,
    "PASS" if gold_session_count == silver_session_count else "FAIL"
))

In [0]:
from pyspark.sql.functions import col, current_timestamp

spark.sql("create schema if not exists ecommerce_project.audit")

orders = spark.table("ecommerce_project.silver.orders")
order_items = spark.table("ecommerce_project.silver.order_items")
sessions = spark.table("ecommerce_project.silver.website_sessions")
fact_sales = spark.table("ecommerce_project.gold.fact_sales")
session_funnel = spark.table("ecommerce_project.gold.session_funnel")


In [0]:
checks = []

# check null order_id
null_order_ids = orders.filter(col("order_id").isNull()).count()

checks.append((
    "Orders: null primary keys",
    null_order_ids,
    0,
    "PASS" if null_order_ids == 0 else "FAIL"))

In [0]:
#check duplicate order_id
duplicate_order_ids = (
    orders.groupBy("order_id")
    .count()
    .filter(col("count") > 1)
    .count()
)

checks.append((
    "Orders: duplicate primary keys",
    duplicate_order_ids,
    0,
    "PASS" if duplicate_order_ids == 0 else "FAIL"
))

In [0]:
#check invalid prices (zero or negative)

invalid_prices = order_items.filter(col("price_usd") <= 0).count()

checks.append((
    "Order items: invalid prices",
    invalid_prices,
    0,
    "PASS" if invalid_prices == 0 else "FAIL"
))


In [0]:
#compare Silver order_items and Gold fact_sales have equal row counts.
silver_item_count = order_items.count()
gold_sales_count = fact_sales.count()

checks.append((
    "Silver items vs Gold sales",
    gold_sales_count,
    silver_item_count,
    "PASS" if gold_sales_count == silver_item_count else "FAIL"
))
#compare Silver sessions and Gold funnel have equal row counts.
silver_session_count = sessions.count()
gold_session_count = session_funnel.count()

checks.append((
    "Silver sessions vs Gold funnel",
    gold_session_count,
    silver_session_count,
    "PASS" if gold_session_count == silver_session_count else "FAIL"
))


In [0]:
# Create results DataFrame
quality_results_df = (
    spark.createDataFrame(
        checks,
        ["check_name", "actual_value", "expected_value", "status"]
    )
    .withColumn("checked_at", current_timestamp())
)

In [0]:
# Save audit table
(
    quality_results_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("ecommerce_project.audit.data_quality_results")
)



In [0]:
display(quality_results_df)